# 🚀 Starboard AI Agent Demo

This notebook demonstrates how to use the **Starboard AI Agent** programmatically
to analyze and optimize Databricks workloads using **multi-turn conversations**.

## What This Demo Covers
1. **Install** — Install the starboard packages
2. **Configure** — Set up credentials and configuration
3. **Analyze** — Ask the agent to analyze your workspace
4. **Inspect** — View execution details and conversation history

## Prerequisites
- Databricks workspace with compute attached (Cluster and DW)
- Model Serving configured in Databricks workspace

## 1️⃣ Install the Starboard CLI Package

Install the Starboard packages including the SDK for multi-turn notebook conversations.

In [ ]:
%sh
  ## Install Starboard AI Agent package
  #############################################################
  pip3 install --quiet \
  "starboard-core @ git+https://github.com/bluewatersql/starboard.git#subdirectory=packages/starboard-core" \
  "starboard @ git+https://github.com/bluewatersql/starboard.git#subdirectory=packages/starboard"

## 2️⃣ Configure Credentials & Settings

The agent requires:
- **Databricks credentials** - Host URL and Personal Access Token (derived fromm the Notebook context)
- **LLM credentials** - Dervied for Databricks ML Serving Endpoints or use an API key for OpenAI compatible provider

In [ ]:
# =============================================================================
# CONFIGURATION
# =============================================================================
import os

# Direct configuration (for demo/development only)
DATABRICKS_HOST = spark.conf.get("spark.databricks.workspaceUrl", "")
DATABRICKS_TOKEN = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()

# LLM Configuration
LLM_MODEL = "databricks-claude-opus-4-8"

LLM_MAX_TOKENS = 64_000
DATABRICKS_WAREHOUSE_ID="<<WAREHOUSE_ID>>"

# Set environment variables for the CLI
os.environ["DATABRICKS_HOST"] = f"https://{DATABRICKS_HOST}"
os.environ["DATABRICKS_TOKEN"] = DATABRICKS_TOKEN
os.environ["DATABRICKS_WAREHOUSE_ID"] = DATABRICKS_WAREHOUSE_ID

os.environ["LLM_BASE_URL"]=f"{os.environ['DATABRICKS_HOST']}/serving-endpoints"
os.environ["LLM_API_KEY"] = DATABRICKS_TOKEN
os.environ["LLM_MODEL"] = LLM_MODEL
os.environ["LLM_MAX_TOKENS"] = str(LLM_MAX_TOKENS)
os.environ["LLM_TEMPERATURE"] = "0.4"

# Discovery
os.environ["DISCOVERY_LLM_MODEL"]="databricks-gpt-5-6-sol"
os.environ["DISCOVERY_LLM_TEMPERATURE"]="0.3"
os.environ["DISCOVERY_MAX_PARALLELISM"]="8"
           
os.environ["DISCOVERY_OUTPUT_DIR"] = "/Volumes/<<CATALOG>>/<<SCHEMA>>/<<VOLUME>>/starboard-discovery"

## 3️⃣ Define Your Conversation

The Starboard Agent supports **multi-turn conversations**. Define your initial
question below. The agent maintains full context
between turns — it remembers discovered entities, prior analysis, and tool results.

Edit the goals below to specify your analysis conversation.

In [ ]:
INITIAL_QUESTION = "workspace discovery"

# Session name controls conversation persistence.
# - Set to a fixed name (e.g. "job-266829") to RESUME a previous conversation.
# - Leave as None to always start fresh (a unique name is generated each run).
SESSION_NAME = None

## 4️⃣ Create a Conversation Session

The SDK manages conversation state automatically. Create a session once, then
send as many questions as you like — context carries forward between turns.

In [ ]:
import json
import logging
from datetime import datetime
from starboard.infra.observability.logging import setup_structured_logging
from starboard.sdk import StarboardClient

# Suppress debug/info output — change to logging.DEBUG to see full traces
setup_structured_logging(level=logging.WARNING)

client = await StarboardClient.from_env()

# Use a fixed SESSION_NAME above to resume a prior conversation,
# or leave it None to generate a unique name and start fresh each run.
_session_name = SESSION_NAME or f"discovery-{datetime.now().strftime('%Y%m%d-%H%M%S')}"
session = await client.create_session(name=_session_name)
print(f"Session: {session.session_name}")

In [ ]:
print("=" * 70)
print("🚀 Workspace Discovery")
print("=" * 70)
print(f"Prompt: {INITIAL_QUESTION}\n")

initial_response = await session.ask(INITIAL_QUESTION, timeout=2700)

if not initial_response.ok:
    print("⚠️  Agent error — check details below:")
    print(f"   Error : {initial_response.error}")
    print()
    print("Possible causes:")
    print("  • Databricks output content filtering redacted a numeric ID (e.g. US_BANK_NUMBER)")
    print("    → Try a different job/query ID, or disable workspace PII output filtering.")
    print("  • The job/query ID does not exist in this workspace.")
    print("  • LLM API authentication or quota issue.")
else:
    print(initial_response.markdown)

In [ ]:
%sh 
ls -r /Volumes/<<CATALOG>>/<<SCHEMA>>/<<VOLUME>>/starboard-discovery/*/*

## 5️⃣  View Structured Output

Each response is an `AgentResponse` with structured fields for programmatic use.

In [ ]:
all_responses = [initial_response]

print("=" * 60)
print("📊 Conversation Summary")
print("=" * 60)
print(f"Session:  {session.session_name}")
print(f"Turns:    {len(all_responses)}")
print()

for i, resp in enumerate(all_responses, start=1):
    q = resp.question[:60] + ("…" if len(resp.question) > 60 else "")
    tokens = f"{resp.tokens_used:,}" if resp.tokens_used else "n/a"
    cost = f"${resp.cost_usd:.4f}" if resp.cost_usd else "n/a"
    print(f"  Turn {i}: {q}")
    print(f"           domain={resp.domain}  tokens={tokens}  "
          f"cost={cost}  latency={resp.duration_seconds:.1f}s")
    print()

total_tokens = sum(r.tokens_used or 0 for r in all_responses)
total_cost = sum(r.cost_usd or 0.0 for r in all_responses)
print(f"  Total tokens: {total_tokens:,}")
print(f"  Total cost:   ${total_cost:.4f}")
print("=" * 60)

## 6️⃣ Cleanup

Close the session and release resources when finished.

In [ ]:
await client.close()
print("Client closed — all resources released.")